In [ ]:
!pip install -q transformers datasets evaluate accelerate sentencepiece rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [ ]:
import os
import time
import math
import json
import random
import warnings
from dataclasses import dataclass

import torch
import pandas as pd
import numpy as np

from datasets import load_dataset
import evaluate

from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")

In [ ]:
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DRAFT_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TARGET_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

MAX_SAMPLES = 50          
MAX_INPUT_CHARS = 4000    
MAX_NEW_TOKENS = 64
TEMPERATURE = 0.0         
TOP_P = 1.0
DO_SAMPLE = False

OUTPUT_DIR = "./spec_decode_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print("DEVICE:", DEVICE)
print("Draft model :", DRAFT_MODEL_NAME)
print("Target model:", TARGET_MODEL_NAME)

DEVICE: cuda
Draft model : Qwen/Qwen2.5-0.5B-Instruct
Target model: Qwen/Qwen2.5-3B-Instruct


In [19]:
rouge = evaluate.load("rouge")

In [20]:
dataset = load_dataset("cnn_dailymail", "3.0.0", split="validation")

records = []
for i, item in enumerate(dataset):
    article = item["article"][:MAX_INPUT_CHARS].strip()
    highlight = item["highlights"].strip()
    if article and highlight:
        records.append({
            "id": i,
            "article": article,
            "reference": highlight
        })
    if len(records) >= MAX_SAMPLES:
        break

df_data = pd.DataFrame(records)
print(df_data.shape)
df_data.head(2)

(50, 3)


,id,article,reference
0,0,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...
1,1,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...


In [21]:
target_tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL_NAME, trust_remote_code=True)
draft_tokenizer = AutoTokenizer.from_pretrained(DRAFT_MODEL_NAME, trust_remote_code=True)

if target_tokenizer.pad_token is None:
    target_tokenizer.pad_token = target_tokenizer.eos_token
if draft_tokenizer.pad_token is None:
    draft_tokenizer.pad_token = draft_tokenizer.eos_token

target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)
target_model.eval()

draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)
draft_model.eval()

print("Models loaded.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Models loaded.


In [22]:
def build_summary_prompt(article: str) -> str:
    return (
        "Summarize the following news article in exactly 2 short sentences. "
        "Do not add extra commentary.\n\n"
        f"Article:\n{article}\n\n"
        "Summary:\n"
    )

In [23]:
@torch.no_grad()
def generate_target_only(
    article: str,
    max_new_tokens: int = MAX_NEW_TOKENS,
    do_sample: bool = DO_SAMPLE,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
):
    prompt = build_summary_prompt(article)

    inputs = target_tokenizer(prompt, return_tensors="pt").to(target_model.device)
    input_len = inputs["input_ids"].shape[1]

    start = time.perf_counter()

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": target_tokenizer.pad_token_id,
        "eos_token_id": target_tokenizer.eos_token_id,
        "do_sample": do_sample,
    }

    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p

    outputs = target_model.generate(**inputs, **gen_kwargs)

    end = time.perf_counter()

    output_ids = outputs[0][input_len:]
    output_text = target_tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    stop_markers = ["Human:", "Assistant:", "\n\nHuman:", "\n\nAssistant:"]
    for marker in stop_markers:
      if marker in output_text:
        output_text = output_text.split(marker)[0].strip()

    latency = end - start
    output_tokens = len(output_ids)

    return {
        "prompt": prompt,
        "prediction": output_text,
        "latency_sec": latency,
        "output_tokens": output_tokens,
        "tokens_per_sec": output_tokens / latency if latency > 0 else None,
    }

In [24]:
sample_article = df_data.iloc[0]["article"]
sample_ref = df_data.iloc[0]["reference"]

result = generate_target_only(sample_article)

print("=== Prediction ===")
print(result["prediction"])
print("\n=== Reference ===")
print(sample_ref)
print("\n=== Stats ===")
print({
    "latency_sec": round(result["latency_sec"], 3),
    "output_tokens": result["output_tokens"],
    "tokens_per_sec": round(result["tokens_per_sec"], 3) if result["tokens_per_sec"] else None
})

=== Prediction ===
Zully Broussard's kidney donation sparked a multi-patient transplant chain through data-driven matching, resulting in six successful transplants; her act of kindness amplified by advanced computational algorithms, enabling previously unattainable organ exchanges.

=== Reference ===
Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

=== Stats ===
{'latency_sec': 0.686, 'output_tokens': 64, 'tokens_per_sec': 93.289}


In [25]:
baseline_results = []

for idx, row in df_data.iterrows():
    out = generate_target_only(row["article"])
    baseline_results.append({
        "id": row["id"],
        "reference": row["reference"],
        "prediction": out["prediction"],
        "latency_sec": out["latency_sec"],
        "output_tokens": out["output_tokens"],
        "tokens_per_sec": out["tokens_per_sec"],
    })

df_baseline = pd.DataFrame(baseline_results)
df_baseline.head(3)

,id,reference,prediction,latency_sec,output_tokens,tokens_per_sec
0,0,Zully Broussard decided to give a kidney to a ...,Zully Broussard's kidney donation sparked a mu...,0.684702,64,93.471326
1,1,The 20th MLS season begins this weekend .\nLea...,"In 1996, Major League Soccer marked its debut ...",0.685449,64,93.369518
2,2,Bafetimbi Gomis collapses within 10 minutes of...,"Swedish striker Bafetimbi Gomis, known for fai...",0.680712,64,94.019189


In [26]:
rouge_scores = rouge.compute(
    predictions=df_baseline["prediction"].tolist(),
    references=df_baseline["reference"].tolist(),
    use_stemmer=True
)

summary_stats = {
    "avg_latency_sec": float(df_baseline["latency_sec"].mean()),
    "avg_output_tokens": float(df_baseline["output_tokens"].mean()),
    "avg_tokens_per_sec": float(df_baseline["tokens_per_sec"].mean()),
    "rouge1": rouge_scores["rouge1"],
    "rouge2": rouge_scores["rouge2"],
    "rougeL": rouge_scores["rougeL"],
    "rougeLsum": rouge_scores["rougeLsum"],
}

summary_stats

{'avg_latency_sec': 0.6807451464800033,
 'avg_output_tokens': 64.0,
 'avg_tokens_per_sec': 94.01993959606175,
 'rouge1': np.float64(0.3100450051696526),
 'rouge2': np.float64(0.0833898361551948),
 'rougeL': np.float64(0.20392345521215965),
 'rougeLsum': np.float64(0.24411605574907363)}

In [27]:
baseline_csv = os.path.join(OUTPUT_DIR, "target_only_baseline_results.csv")
baseline_json = os.path.join(OUTPUT_DIR, "target_only_baseline_summary.json")

df_baseline.to_csv(baseline_csv, index=False)

with open(baseline_json, "w", encoding="utf-8") as f:
    json.dump(summary_stats, f, ensure_ascii=False, indent=2)

print("Saved to:")
print(baseline_csv)
print(baseline_json)

Saved to:
./spec_decode_outputs/target_only_baseline_results.csv
./spec_decode_outputs/target_only_baseline_summary.json


In [28]:
for i in range(3):
    print(f"===== Sample {i} =====")
    print("Prediction:")
    print(df_baseline.iloc[i]["prediction"])
    print("\nReference:")
    print(df_baseline.iloc[i]["reference"])
    print("\n")

===== Sample 0 =====
Prediction:
Zully Broussard's kidney donation sparked a multi-patient transplant chain through data-driven matching, resulting in six successful transplants; her act of kindness amplified by advanced computational algorithms, enabling previously unattainable organ exchanges.

Reference:
Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .


===== Sample 1 =====
Prediction:
In 1996, Major League Soccer marked its debut with a historic match in San Jose, California, heralding a new era for American soccer. Despite initial challenges and financial struggles, the league has grown significantly, with increased attendance, expanded teams, and rising popularity, now poised to potentially become the second-most

Reference:
The 20th MLS season begins this weekend .
League has changed dramatically since its inception in 1996 .
Some question whether rules regarding salary caps and transf

In [29]:
# Cell 12：辅助函数——单步 greedy next token
import torch
import time

@torch.no_grad()
def get_next_token_greedy(model, input_ids, attention_mask=None):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
    return next_token

In [30]:
# Cell 13：draft model 连续生成 k 个 token
@torch.no_grad()
def draft_generate_k_tokens(model, input_ids, k):
    """
    用 draft model 从当前 input_ids 开始，greedy 连续生成 k 个 token
    返回:
      draft_tokens: [1, k]
    """
    cur_ids = input_ids.clone()
    draft_tokens = []

    for _ in range(k):
        next_token = get_next_token_greedy(model, cur_ids)
        draft_tokens.append(next_token)
        cur_ids = torch.cat([cur_ids, next_token], dim=1)

    draft_tokens = torch.cat(draft_tokens, dim=1)  # [1, k]
    return draft_tokens

In [31]:
# Cell 14：target model 验证 draft token 前缀
@torch.no_grad()
def verify_draft_tokens(target_model, prefix_ids, draft_tokens):
    """
    target model 从 prefix_ids 开始，逐步 greedy 验证 draft_tokens
    返回:
      accepted_tokens: list[int]
      accepted_count: int
      next_target_token: torch.Tensor shape [1, 1]
    """
    cur_ids = prefix_ids.clone()
    accepted = []

    for i in range(draft_tokens.shape[1]):
        target_next = get_next_token_greedy(target_model, cur_ids)  # [1,1]
        draft_tok = draft_tokens[:, i:i+1]  # [1,1]

        if torch.equal(target_next, draft_tok):
            accepted.append(int(draft_tok.item()))
            cur_ids = torch.cat([cur_ids, draft_tok], dim=1)
        else:
            # draft 在这里失败，target_next 作为真正下一个 token
            return accepted, len(accepted), target_next

    # 如果 draft 全通过，再额外让 target 生成 1 个新 token
    next_target_token = get_next_token_greedy(target_model, cur_ids)
    return accepted, len(accepted), next_target_token

In [32]:
# Cell 15：fixed-k 生成函数
@torch.no_grad()
def generate_fixed_k(
    article: str,
    k: int = 4,
    max_new_tokens: int = MAX_NEW_TOKENS,
):
    prompt = build_summary_prompt(article)

    inputs = target_tokenizer(prompt, return_tensors="pt").to(target_model.device)
    input_ids = inputs["input_ids"]

    generated = []
    total_accepted = 0
    total_drafted = 0

    start = time.perf_counter()

    for _ in range(max_new_tokens):
        # 1) draft model 先猜 k 个 token
        draft_inputs = input_ids.to(draft_model.device)
        draft_tokens = draft_generate_k_tokens(draft_model, draft_inputs, k=k)  # [1, k]
        total_drafted += k

        # 2) target model 验证
        accepted, accepted_count, next_target_token = verify_draft_tokens(
            target_model,
            input_ids.to(target_model.device),
            draft_tokens.to(target_model.device)
        )

        # 接受的 token 先写入
        for tok in accepted:
            tok_tensor = torch.tensor([[tok]], device=target_model.device)
            input_ids = torch.cat([input_ids.to(target_model.device), tok_tensor], dim=1)
            generated.append(tok)

        total_accepted += accepted_count

        # 再写入 target 的下一个真实 token
        input_ids = torch.cat([input_ids.to(target_model.device), next_target_token], dim=1)
        generated.append(int(next_target_token.item()))

        # 如果到 eos 就停
        if int(next_target_token.item()) == target_tokenizer.eos_token_id:
            break

        if len(generated) >= max_new_tokens:
            break

    end = time.perf_counter()

    output_ids = generated[:max_new_tokens]
    output_text = target_tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    stop_markers = ["Human:", "Assistant:", "\n\nHuman:", "\n\nAssistant:"]
    for marker in stop_markers:
        if marker in output_text:
            output_text = output_text.split(marker)[0].strip()

    latency = end - start
    output_tokens = len(output_ids)
    acceptance_rate = total_accepted / total_drafted if total_drafted > 0 else 0.0

    return {
        "prompt": prompt,
        "prediction": output_text,
        "latency_sec": latency,
        "output_tokens": output_tokens,
        "tokens_per_sec": output_tokens / latency if latency > 0 else None,
        "acceptance_rate": acceptance_rate,
        "accepted_tokens": total_accepted,
        "drafted_tokens": total_drafted,
    }

In [33]:
# Cell 16：单条测试 fixed-k
sample_article = df_data.iloc[0]["article"]
sample_ref = df_data.iloc[0]["reference"]

result_fixed = generate_fixed_k(sample_article, k=4)

print("=== Prediction (fixed-k) ===")
print(result_fixed["prediction"])
print("\n=== Reference ===")
print(sample_ref)
print("\n=== Stats ===")
print({
    "latency_sec": round(result_fixed["latency_sec"], 3),
    "output_tokens": result_fixed["output_tokens"],
    "tokens_per_sec": round(result_fixed["tokens_per_sec"], 3) if result_fixed["tokens_per_sec"] else None,
    "acceptance_rate": round(result_fixed["acceptance_rate"], 3),
    "accepted_tokens": result_fixed["accepted_tokens"],
    "drafted_tokens": result_fixed["drafted_tokens"],
})

=== Prediction (fixed-k) ===
Zully Broussard's kidney donation led to a six-patient kidney transplant chain, facilitated by data processing of genetic profiles, surprising her with the potential to extend lives beyond her initial recipient. The process, made possible by David Jacobs' MatchGrid program, has since been streamlined, allowing for rapid matching of compatible

=== Reference ===
Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

=== Stats ===
{'latency_sec': 2.783, 'output_tokens': 64, 'tokens_per_sec': 22.996, 'acceptance_rate': 0.266, 'accepted_tokens': 33, 'drafted_tokens': 124}


In [36]:
fixed_k_results = []

for idx, row in df_data.iterrows():
    out = generate_fixed_k(row["article"], k=2)
    fixed_k_results.append({
        "id": row["id"],
        "reference": row["reference"],
        "prediction": out["prediction"],
        "latency_sec": out["latency_sec"],
        "output_tokens": out["output_tokens"],
        "tokens_per_sec": out["tokens_per_sec"],
        "acceptance_rate": out["acceptance_rate"],
        "accepted_tokens": out["accepted_tokens"],
        "drafted_tokens": out["drafted_tokens"],
    })

df_fixed_k = pd.DataFrame(fixed_k_results)
df_fixed_k.head(3)

,id,reference,prediction,latency_sec,output_tokens,tokens_per_sec,acceptance_rate,accepted_tokens,drafted_tokens
0,0,Zully Broussard decided to give a kidney to a ...,Zully Broussard's kidney donation led to a six...,2.319022,64,27.597841,0.414286,29,70
1,1,The 20th MLS season begins this weekend .\nLea...,"On April 6, 1996, Major League Soccer (MLS) ma...",2.274575,64,28.137119,0.583333,35,60
2,2,Bafetimbi Gomis collapses within 10 minutes of...,"French striker Bafetimbi Gomis, who has a hist...",1.951535,64,32.794695,0.583333,35,60


In [37]:
rouge_scores_fixed = rouge.compute(
    predictions=df_fixed_k["prediction"].tolist(),
    references=df_fixed_k["reference"].tolist(),
    use_stemmer=True
)

fixed_k_stats = {
    "avg_latency_sec": float(df_fixed_k["latency_sec"].mean()),
    "avg_output_tokens": float(df_fixed_k["output_tokens"].mean()),
    "avg_tokens_per_sec": float(df_fixed_k["tokens_per_sec"].mean()),
    "avg_acceptance_rate": float(df_fixed_k["acceptance_rate"].mean()),
    "rouge1": rouge_scores_fixed["rouge1"],
    "rouge2": rouge_scores_fixed["rouge2"],
    "rougeL": rouge_scores_fixed["rougeL"],
    "rougeLsum": rouge_scores_fixed["rougeLsum"],
}

fixed_k_stats

{'avg_latency_sec': 1.8227813146200242,
 'avg_output_tokens': 64.0,
 'avg_tokens_per_sec': 36.23462245676679,
 'avg_acceptance_rate': 0.5621234930171496,
 'rouge1': np.float64(0.31620256978377437),
 'rouge2': np.float64(0.0995451761624395),
 'rougeL': np.float64(0.20757584794693512),
 'rougeLsum': np.float64(0.248638752417641)}

In [ ]:
# 把当前 fixed-k 的结果手动保存为 k=2 版本，避免后续被覆盖
fixed_k2_stats = fixed_k_stats.copy()
df_fixed_k2 = df_fixed_k.copy()

In [ ]:
#C19 定义 adaptive-k 的超参数，控制初始 draft 长度、上下界以及调整阈值
ADAPTIVE_K_INIT = 2
ADAPTIVE_K_MIN = 1
ADAPTIVE_K_MAX = 4

ADAPTIVE_WINDOW = 4          # 最近多少轮 acceptance 作为参考
ADAPTIVE_HIGH_THRES = 0.80   # 最近 acceptance 高于该阈值则增大 k
ADAPTIVE_LOW_THRES = 0.45    # 最近 acceptance 低于该阈值则减小 k

print({
    "k_init": ADAPTIVE_K_INIT,
    "k_min": ADAPTIVE_K_MIN,
    "k_max": ADAPTIVE_K_MAX,
    "window": ADAPTIVE_WINDOW,
    "high_thres": ADAPTIVE_HIGH_THRES,
    "low_thres": ADAPTIVE_LOW_THRES
})

{'k_init': 2, 'k_min': 1, 'k_max': 4, 'window': 4, 'high_thres': 0.8, 'low_thres': 0.45}


In [ ]:
#C20 增强版 adaptive-k 更新规则，升 k 更谨慎，降 k 更果断，减少前期误判带来的震荡
def update_adaptive_k(
    k_current,
    acceptance_history,
    k_min=ADAPTIVE_K_MIN,
    k_max=ADAPTIVE_K_MAX,
    window=ADAPTIVE_WINDOW,
    high_thres=ADAPTIVE_HIGH_THRES,
    low_thres=ADAPTIVE_LOW_THRES,
):
    if len(acceptance_history) == 0:
        return k_current

    recent = acceptance_history[-window:]
    avg_recent = sum(recent) / len(recent)

    # 规则1：如果最近窗口平均 acceptance 太低，则快速降 k
    if avg_recent < low_thres:
        return max(k_current - 1, k_min)

    # 规则2：只有最近两轮 acceptance 都很高，且窗口平均也足够高，才升 k
    if len(acceptance_history) >= 2:
        last_two = acceptance_history[-2:]
        if min(last_two) >= 0.8 and avg_recent >= 0.7:
            return min(k_current + 1, k_max)

    # 否则保持不变
    return k_current

In [ ]:
#C21 实现 adaptive-k 的两阶段生成函数，记录每轮动态 k 和 acceptance 统计
@torch.no_grad()
def generate_adaptive_k(
    article: str,
    k_init: int = ADAPTIVE_K_INIT,
    k_min: int = ADAPTIVE_K_MIN,
    k_max: int = ADAPTIVE_K_MAX,
    window: int = ADAPTIVE_WINDOW,
    high_thres: float = ADAPTIVE_HIGH_THRES,
    low_thres: float = ADAPTIVE_LOW_THRES,
    max_new_tokens: int = MAX_NEW_TOKENS,
):
    prompt = build_summary_prompt(article)

    inputs = target_tokenizer(prompt, return_tensors="pt").to(target_model.device)
    input_ids = inputs["input_ids"]

    generated = []
    total_accepted = 0
    total_drafted = 0

    k_current = k_init
    acceptance_history = []
    k_history = []

    start = time.perf_counter()

    while len(generated) < max_new_tokens:
        k_history.append(k_current)

        # 1) draft model 先生成 k_current 个 token
        draft_inputs = input_ids.to(draft_model.device)
        draft_tokens = draft_generate_k_tokens(draft_model, draft_inputs, k=k_current)
        total_drafted += k_current

        # 2) target model 验证
        accepted, accepted_count, next_target_token = verify_draft_tokens(
            target_model,
            input_ids.to(target_model.device),
            draft_tokens.to(target_model.device)
        )

        # 3) 写入所有被接受的 token
        for tok in accepted:
            tok_tensor = torch.tensor([[tok]], device=target_model.device)
            input_ids = torch.cat([input_ids.to(target_model.device), tok_tensor], dim=1)
            generated.append(tok)

            if len(generated) >= max_new_tokens:
                break

        total_accepted += accepted_count

        # 当前轮 acceptance rate
        round_acceptance = accepted_count / k_current if k_current > 0 else 0.0
        acceptance_history.append(round_acceptance)

        if len(generated) >= max_new_tokens:
            break

        # 4) 再写入 target 生成的下一个真实 token
        input_ids = torch.cat([input_ids.to(target_model.device), next_target_token], dim=1)
        generated.append(int(next_target_token.item()))

        # 如果遇到 eos，就结束
        if int(next_target_token.item()) == target_tokenizer.eos_token_id:
            break

        # 5) 更新下一轮 k
        k_current = update_adaptive_k(
            k_current=k_current,
            acceptance_history=acceptance_history,
            k_min=k_min,
            k_max=k_max,
            window=window,
            high_thres=high_thres,
            low_thres=low_thres,
        )

    end = time.perf_counter()

    output_ids = generated[:max_new_tokens]
    output_text = target_tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    stop_markers = ["Human:", "Assistant:", "\n\nHuman:", "\n\nAssistant:"]
    for marker in stop_markers:
        if marker in output_text:
            output_text = output_text.split(marker)[0].strip()

    latency = end - start
    output_tokens = len(output_ids)
    acceptance_rate = total_accepted / total_drafted if total_drafted > 0 else 0.0
    avg_k = sum(k_history) / len(k_history) if len(k_history) > 0 else 0.0

    return {
        "prompt": prompt,
        "prediction": output_text,
        "latency_sec": latency,
        "output_tokens": output_tokens,
        "tokens_per_sec": output_tokens / latency if latency > 0 else None,
        "acceptance_rate": acceptance_rate,
        "accepted_tokens": total_accepted,
        "drafted_tokens": total_drafted,
        "avg_k": avg_k,
        "k_history": k_history,
        "acceptance_history": acceptance_history,
    }

In [50]:
#C22 中文注释：对单条样本测试 adaptive-k 是否能正常运行，并查看动态 k 的变化情况
sample_article = df_data.iloc[0]["article"]
sample_ref = df_data.iloc[0]["reference"]

result_adaptive = generate_adaptive_k(sample_article)

print("=== Prediction (adaptive-k) ===")
print(result_adaptive["prediction"])
print("\n=== Reference ===")
print(sample_ref)
print("\n=== Stats ===")
print({
    "latency_sec": round(result_adaptive["latency_sec"], 3),
    "output_tokens": result_adaptive["output_tokens"],
    "tokens_per_sec": round(result_adaptive["tokens_per_sec"], 3) if result_adaptive["tokens_per_sec"] else None,
    "acceptance_rate": round(result_adaptive["acceptance_rate"], 3),
    "accepted_tokens": result_adaptive["accepted_tokens"],
    "drafted_tokens": result_adaptive["drafted_tokens"],
    "avg_k": round(result_adaptive["avg_k"], 3),
})

print("\n=== k history ===")
print(result_adaptive["k_history"])

print("\n=== acceptance history ===")
print([round(x, 3) for x in result_adaptive["acceptance_history"]])

=== Prediction (adaptive-k) ===
Zully Broussard's kidney donation led to a six-patient kidney transplant chain, facilitated by data processing of genetic profiles, surprising her with the potential to extend lives beyond her initial recipient. The process, made possible by David Jacobs' MatchGrid program, has since been streamlined, allowing for rapid matching of compatible

=== Reference ===
Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

=== Stats ===
{'latency_sec': 2.27, 'output_tokens': 64, 'tokens_per_sec': 28.197, 'acceptance_rate': 0.422, 'accepted_tokens': 27, 'drafted_tokens': 64, 'avg_k': 1.73}

=== k history ===
[2, 2, 3, 3, 3, 3, 2, 1, 1, 1, 2, 3, 3, 3, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 3, 3, 3, 2, 1, 1]

=== acceptance history ===
[1.0, 1.0, 0.667, 0.0, 0.667, 0.0, 0.5, 1.0, 1.0, 1.0, 1.0, 0.333, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 

In [ ]:
#C23 对当前样本集批量运行 adaptive-k，并保存逐样本结果到 DataFrame
adaptive_results = []

for idx, row in df_data.iterrows():
    out = generate_adaptive_k(row["article"])
    adaptive_results.append({
        "id": row["id"],
        "reference": row["reference"],
        "prediction": out["prediction"],
        "latency_sec": out["latency_sec"],
        "output_tokens": out["output_tokens"],
        "tokens_per_sec": out["tokens_per_sec"],
        "acceptance_rate": out["acceptance_rate"],
        "accepted_tokens": out["accepted_tokens"],
        "drafted_tokens": out["drafted_tokens"],
        "avg_k": out["avg_k"],
    })

df_adaptive = pd.DataFrame(adaptive_results)
df_adaptive.head(3)

,id,reference,prediction,latency_sec,output_tokens,tokens_per_sec,acceptance_rate,accepted_tokens,drafted_tokens,avg_k
0,0,Zully Broussard decided to give a kidney to a ...,Zully Broussard's kidney donation led to a six...,2.268941,64,28.206993,0.421875,27,64,1.729730
1,1,The 20th MLS season begins this weekend .\nLea...,"On April 6, 1996, Major League Soccer (MLS) ma...",2.314850,64,27.647574,0.569231,37,65,2.321429
2,2,Bafetimbi Gomis collapses within 10 minutes of...,"French striker Bafetimbi Gomis, who has a hist...",1.967448,64,32.529446,0.515152,34,66,2.200000


In [ ]:
#C24 计算 adaptive-k 的整体质量和效率指标，作为后续对比的核心结果
rouge_scores_adaptive = rouge.compute(
    predictions=df_adaptive["prediction"].tolist(),
    references=df_adaptive["reference"].tolist(),
    use_stemmer=True
)

adaptive_stats = {
    "avg_latency_sec": float(df_adaptive["latency_sec"].mean()),
    "avg_output_tokens": float(df_adaptive["output_tokens"].mean()),
    "avg_tokens_per_sec": float(df_adaptive["tokens_per_sec"].mean()),
    "avg_acceptance_rate": float(df_adaptive["acceptance_rate"].mean()),
    "avg_k": float(df_adaptive["avg_k"].mean()),
    "rouge1": rouge_scores_adaptive["rouge1"],
    "rouge2": rouge_scores_adaptive["rouge2"],
    "rougeL": rouge_scores_adaptive["rougeL"],
    "rougeLsum": rouge_scores_adaptive["rougeLsum"],
}

adaptive_stats

{'avg_latency_sec': 1.8353312575399285,
 'avg_output_tokens': 64.0,
 'avg_tokens_per_sec': 35.840287793083505,
 'avg_acceptance_rate': 0.5164533685879917,
 'avg_k': 2.083662513063082,
 'rouge1': np.float64(0.31620256978377437),
 'rouge2': np.float64(0.0995451761624395),
 'rougeL': np.float64(0.20757584794693512),
 'rougeLsum': np.float64(0.248638752417641)}

In [ ]:
#C25 整理多种方法的核心指标，形成最终可用于项目分析的对比表
comparison_rows = []

# target-only
comparison_rows.append({
    "method": "target_only",
    "avg_latency_sec": summary_stats["avg_latency_sec"],
    "avg_tokens_per_sec": summary_stats["avg_tokens_per_sec"],
    "avg_acceptance_rate": None,
    "avg_k": None,
    "rouge1": summary_stats["rouge1"],
    "rouge2": summary_stats["rouge2"],
    "rougeL": summary_stats["rougeL"],
    "rougeLsum": summary_stats["rougeLsum"],
})

if "fixed_k4_stats" in globals():
    comparison_rows.append({
        "method": "fixed_k_4",
        "avg_latency_sec": fixed_k4_stats["avg_latency_sec"],
        "avg_tokens_per_sec": fixed_k4_stats["avg_tokens_per_sec"],
        "avg_acceptance_rate": fixed_k4_stats["avg_acceptance_rate"],
        "avg_k": 4.0,
        "rouge1": fixed_k4_stats["rouge1"],
        "rouge2": fixed_k4_stats["rouge2"],
        "rougeL": fixed_k4_stats["rougeL"],
        "rougeLsum": fixed_k4_stats["rougeLsum"],
    })

if "fixed_k2_stats" in globals():
    comparison_rows.append({
        "method": "fixed_k_2",
        "avg_latency_sec": fixed_k2_stats["avg_latency_sec"],
        "avg_tokens_per_sec": fixed_k2_stats["avg_tokens_per_sec"],
        "avg_acceptance_rate": fixed_k2_stats["avg_acceptance_rate"],
        "avg_k": 2.0,
        "rouge1": fixed_k2_stats["rouge1"],
        "rouge2": fixed_k2_stats["rouge2"],
        "rougeL": fixed_k2_stats["rougeL"],
        "rougeLsum": fixed_k2_stats["rougeLsum"],
    })

elif "fixed_k_stats" in globals():
    comparison_rows.append({
        "method": "fixed_k_current",
        "avg_latency_sec": fixed_k_stats["avg_latency_sec"],
        "avg_tokens_per_sec": fixed_k_stats["avg_tokens_per_sec"],
        "avg_acceptance_rate": fixed_k_stats["avg_acceptance_rate"],
        "avg_k": None,
        "rouge1": fixed_k_stats["rouge1"],
        "rouge2": fixed_k_stats["rouge2"],
        "rougeL": fixed_k_stats["rougeL"],
        "rougeLsum": fixed_k_stats["rougeLsum"],
    })

# adaptive
comparison_rows.append({
    "method": "adaptive_k",
    "avg_latency_sec": adaptive_stats["avg_latency_sec"],
    "avg_tokens_per_sec": adaptive_stats["avg_tokens_per_sec"],
    "avg_acceptance_rate": adaptive_stats["avg_acceptance_rate"],
    "avg_k": adaptive_stats["avg_k"],
    "rouge1": adaptive_stats["rouge1"],
    "rouge2": adaptive_stats["rouge2"],
    "rougeL": adaptive_stats["rougeL"],
    "rougeLsum": adaptive_stats["rougeLsum"],
})

df_compare = pd.DataFrame(comparison_rows)
df_compare

In [ ]:
#A 把 draft model 从 0.5B 切换到 1.5B，target model 保持 3B 不变
DRAFT_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
TARGET_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Draft model :", DRAFT_MODEL_NAME)
print("Target model:", TARGET_MODEL_NAME)

Draft model : Qwen/Qwen2.5-1.5B-Instruct
Target model: Qwen/Qwen2.5-3B-Instruct


In [ ]:
#B 重新加载新的 draft model（1.5B）和 target model（3B），确保 tokenizer 与 device 状态一致
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc
import torch

try:
    del draft_model
    del draft_tokenizer
except:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

target_tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL_NAME, trust_remote_code=True)
draft_tokenizer = AutoTokenizer.from_pretrained(DRAFT_MODEL_NAME, trust_remote_code=True)

if target_tokenizer.pad_token is None:
    target_tokenizer.pad_token = target_tokenizer.eos_token
if draft_tokenizer.pad_token is None:
    draft_tokenizer.pad_token = draft_tokenizer.eos_token

# target model 重新加载
try:
    del target_model
except:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)
target_model.eval()

draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)
draft_model.eval()

print("Models reloaded successfully.")
print("Target device map:", getattr(target_model, "hf_device_map", "N/A"))
print("Draft device map :", getattr(draft_model, "hf_device_map", "N/A"))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Models reloaded successfully.
Target device map: N/A
Draft device map : N/A


In [ ]:
#C 先用新 draft model（1.5B）对 fixed-k=2 做单条测试，观察 acceptance 是否提升
sample_article = df_data.iloc[0]["article"]
sample_ref = df_data.iloc[0]["reference"]

result_fixed_k2_15b = generate_fixed_k(sample_article, k=2)

print("=== Prediction (fixed-k=2, draft=1.5B) ===")
print(result_fixed_k2_15b["prediction"])
print("\n=== Reference ===")
print(sample_ref)
print("\n=== Stats ===")
print({
    "latency_sec": round(result_fixed_k2_15b["latency_sec"], 3),
    "output_tokens": result_fixed_k2_15b["output_tokens"],
    "tokens_per_sec": round(result_fixed_k2_15b["tokens_per_sec"], 3) if result_fixed_k2_15b["tokens_per_sec"] else None,
    "acceptance_rate": round(result_fixed_k2_15b["acceptance_rate"], 3),
    "accepted_tokens": result_fixed_k2_15b["accepted_tokens"],
    "drafted_tokens": result_fixed_k2_15b["drafted_tokens"],
})

=== Prediction (fixed-k=2, draft=1.5B) ===
Zully Broussard's kidney donation led to a six-patient kidney transplant chain, facilitated by data processing of genetic profiles, surprising her with the potential to extend lives beyond her initial recipient. The process, made possible by David Jacobs' MatchGrid program, has since been streamlined, allowing for rapid matching of compatible

=== Reference ===
Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

=== Stats ===
{'latency_sec': 3.018, 'output_tokens': 64, 'tokens_per_sec': 21.204, 'acceptance_rate': 0.389, 'accepted_tokens': 28, 'drafted_tokens': 72}


In [ ]:
#D 使用新的 draft model（1.5B）批量运行 fixed-k=2，并保存结果
fixed_k2_15b_results = []

for idx, row in df_data.iterrows():
    out = generate_fixed_k(row["article"], k=2)
    fixed_k2_15b_results.append({
        "id": row["id"],
        "reference": row["reference"],
        "prediction": out["prediction"],
        "latency_sec": out["latency_sec"],
        "output_tokens": out["output_tokens"],
        "tokens_per_sec": out["tokens_per_sec"],
        "acceptance_rate": out["acceptance_rate"],
        "accepted_tokens": out["accepted_tokens"],
        "drafted_tokens": out["drafted_tokens"],
    })

df_fixed_k2_15b = pd.DataFrame(fixed_k2_15b_results)
df_fixed_k2_15b.head(3)

,id,reference,prediction,latency_sec,output_tokens,tokens_per_sec,acceptance_rate,accepted_tokens,drafted_tokens
0,0,Zully Broussard decided to give a kidney to a ...,Zully Broussard's kidney donation led to a six...,3.015034,64,21.226959,0.388889,28,72
1,1,The 20th MLS season begins this weekend .\nLea...,"On April 6, 1996, Major League Soccer (MLS) ma...",2.712744,64,23.592345,0.750000,39,52
2,2,Bafetimbi Gomis collapses within 10 minutes of...,"French striker Bafetimbi Gomis, who has a hist...",2.262543,64,28.286751,0.583333,35,60


In [ ]:
#E 计算新的 fixed-k=2（1.5B -> 3B）整体指标
rouge_scores_fixed_k2_15b = rouge.compute(
    predictions=df_fixed_k2_15b["prediction"].tolist(),
    references=df_fixed_k2_15b["reference"].tolist(),
    use_stemmer=True
)

fixed_k2_15b_stats = {
    "avg_latency_sec": float(df_fixed_k2_15b["latency_sec"].mean()),
    "avg_output_tokens": float(df_fixed_k2_15b["output_tokens"].mean()),
    "avg_tokens_per_sec": float(df_fixed_k2_15b["tokens_per_sec"].mean()),
    "avg_acceptance_rate": float(df_fixed_k2_15b["acceptance_rate"].mean()),
    "rouge1": rouge_scores_fixed_k2_15b["rouge1"],
    "rouge2": rouge_scores_fixed_k2_15b["rouge2"],
    "rougeL": rouge_scores_fixed_k2_15b["rougeL"],
    "rougeLsum": rouge_scores_fixed_k2_15b["rougeLsum"],
}

fixed_k2_15b_stats

{'avg_latency_sec': 2.130592366860037,
 'avg_output_tokens': 64.0,
 'avg_tokens_per_sec': 31.490546472223755,
 'avg_acceptance_rate': 0.6194647109487046,
 'rouge1': np.float64(0.31620256978377437),
 'rouge2': np.float64(0.0995451761624395),
 'rougeL': np.float64(0.20757584794693512),
 'rougeLsum': np.float64(0.248638752417641)}

In [ ]:
#F 先用新 draft model（1.5B）对 adaptive-k 做单条测试，观察动态 k 和 acceptance 表现
sample_article = df_data.iloc[0]["article"]
sample_ref = df_data.iloc[0]["reference"]

result_adaptive_15b = generate_adaptive_k(sample_article)

print("=== Prediction (adaptive-k, draft=1.5B) ===")
print(result_adaptive_15b["prediction"])
print("\n=== Reference ===")
print(sample_ref)
print("\n=== Stats ===")
print({
    "latency_sec": round(result_adaptive_15b["latency_sec"], 3),
    "output_tokens": result_adaptive_15b["output_tokens"],
    "tokens_per_sec": round(result_adaptive_15b["tokens_per_sec"], 3) if result_adaptive_15b["tokens_per_sec"] else None,
    "acceptance_rate": round(result_adaptive_15b["acceptance_rate"], 3),
    "accepted_tokens": result_adaptive_15b["accepted_tokens"],
    "drafted_tokens": result_adaptive_15b["drafted_tokens"],
    "avg_k": round(result_adaptive_15b["avg_k"], 3),
})

print("\n=== k history ===")
print(result_adaptive_15b["k_history"])

print("\n=== acceptance history ===")
print([round(x, 3) for x in result_adaptive_15b["acceptance_history"]])

In [ ]:
#G 使用新的 draft model（1.5B）批量运行 adaptive-k，并保存结果
adaptive_15b_results = []

for idx, row in df_data.iterrows():
    out = generate_adaptive_k(row["article"])
    adaptive_15b_results.append({
        "id": row["id"],
        "reference": row["reference"],
        "prediction": out["prediction"],
        "latency_sec": out["latency_sec"],
        "output_tokens": out["output_tokens"],
        "tokens_per_sec": out["tokens_per_sec"],
        "acceptance_rate": out["acceptance_rate"],
        "accepted_tokens": out["accepted_tokens"],
        "drafted_tokens": out["drafted_tokens"],
        "avg_k": out["avg_k"],
    })

df_adaptive_15b = pd.DataFrame(adaptive_15b_results)
df_adaptive_15b.head(3)

In [ ]:
#H 计算新的 adaptive-k（1.5B -> 3B）整体指标
rouge_scores_adaptive_15b = rouge.compute(
    predictions=df_adaptive_15b["prediction"].tolist(),
    references=df_adaptive_15b["reference"].tolist(),
    use_stemmer=True
)

adaptive_15b_stats = {
    "avg_latency_sec": float(df_adaptive_15b["latency_sec"].mean()),
    "avg_output_tokens": float(df_adaptive_15b["output_tokens"].mean()),
    "avg_tokens_per_sec": float(df_adaptive_15b["tokens_per_sec"].mean()),
    "avg_acceptance_rate": float(df_adaptive_15b["acceptance_rate"].mean()),
    "avg_k": float(df_adaptive_15b["avg_k"].mean()),
    "rouge1": rouge_scores_adaptive_15b["rouge1"],
    "rouge2": rouge_scores_adaptive_15b["rouge2"],
    "rougeL": rouge_scores_adaptive_15b["rougeL"],
    "rougeLsum": rouge_scores_adaptive_15b["rougeLsum"],
}

adaptive_15b_stats

In [ ]:
#I 对比不同 draft model 规模（0.5B vs 1.5B）下 fixed-k=2 和 adaptive-k 的表现
comparison_rows = []

# target-only baseline
comparison_rows.append({
    "method": "target_only",
    "draft_model": "-",
    "avg_latency_sec": summary_stats["avg_latency_sec"],
    "avg_tokens_per_sec": summary_stats["avg_tokens_per_sec"],
    "avg_acceptance_rate": None,
    "avg_k": None,
    "rouge1": summary_stats["rouge1"],
    "rouge2": summary_stats["rouge2"],
    "rougeL": summary_stats["rougeL"],
    "rougeLsum": summary_stats["rougeLsum"],
})

# fixed-k=2 with 0.5B draft
comparison_rows.append({
    "method": "fixed_k_2",
    "draft_model": "Qwen2.5-0.5B",
    "avg_latency_sec": fixed_k2_stats["avg_latency_sec"],
    "avg_tokens_per_sec": fixed_k2_stats["avg_tokens_per_sec"],
    "avg_acceptance_rate": fixed_k2_stats["avg_acceptance_rate"],
    "avg_k": 2.0,
    "rouge1": fixed_k2_stats["rouge1"],
    "rouge2": fixed_k2_stats["rouge2"],
    "rougeL": fixed_k2_stats["rougeL"],
    "rougeLsum": fixed_k2_stats["rougeLsum"],
})

# adaptive with 0.5B draft
comparison_rows.append({
    "method": "adaptive_k",
    "draft_model": "Qwen2.5-0.5B",
    "avg_latency_sec": adaptive_stats["avg_latency_sec"],
    "avg_tokens_per_sec": adaptive_stats["avg_tokens_per_sec"],
    "avg_acceptance_rate": adaptive_stats["avg_acceptance_rate"],
    "avg_k": adaptive_stats["avg_k"],
    "rouge1": adaptive_stats["rouge1"],
    "rouge2": adaptive_stats["rouge2"],
    "rougeL": adaptive_stats["rougeL"],
    "rougeLsum": adaptive_stats["rougeLsum"],
})

# fixed-k=2 with 1.5B draft
comparison_rows.append({
    "method": "fixed_k_2",
    "draft_model": "Qwen2.5-1.5B",
    "avg_latency_sec": fixed_k2_15b_stats["avg_latency_sec"],
    "avg_tokens_per_sec": fixed_k2_15b_stats["avg_tokens_per_sec"],
    "avg_acceptance_rate": fixed_k2_15b_stats["avg_acceptance_rate"],
    "avg_k": 2.0,
    "rouge1": fixed_k2_15b_stats["rouge1"],
    "rouge2": fixed_k2_15b_stats["rouge2"],
    "rougeL": fixed_k2_15b_stats["rougeL"],
    "rougeLsum": fixed_k2_15b_stats["rougeLsum"],
})

# adaptive with 1.5B draft
comparison_rows.append({
    "method": "adaptive_k",
    "draft_model": "Qwen2.5-1.5B",
    "avg_latency_sec": adaptive_15b_stats["avg_latency_sec"],
    "avg_tokens_per_sec": adaptive_15b_stats["avg_tokens_per_sec"],
    "avg_acceptance_rate": adaptive_15b_stats["avg_acceptance_rate"],
    "avg_k": adaptive_15b_stats["avg_k"],
    "rouge1": adaptive_15b_stats["rouge1"],
    "rouge2": adaptive_15b_stats["rouge2"],
    "rougeL": adaptive_15b_stats["rougeL"],
    "rougeLsum": adaptive_15b_stats["rougeLsum"],
})

df_compare_draft_scale = pd.DataFrame(comparison_rows)
df_compare_draft_scale

In [ ]:
# 定义分阶段 adaptive controller 的超参数，不同生成阶段采用不同升降策略
PHASE_K_INIT = 2
PHASE_K_MIN = 1
PHASE_K_MAX = 4

PHASE_WINDOW = 4

# 阶段划分：按已生成 token 占 max_new_tokens 的比例划分
EARLY_RATIO = 0.30
MID_RATIO = 0.70

# early phase：前30%
EARLY_HIGH_THRES = 0.75
EARLY_LOW_THRES = 0.45

# middle phase：30%~70%
MID_HIGH_THRES = 0.85
MID_LOW_THRES = 0.50

# late phase：70%以后
LATE_LOW_THRES = 0.55

print({
    "k_init": PHASE_K_INIT,
    "k_min": PHASE_K_MIN,
    "k_max": PHASE_K_MAX,
    "window": PHASE_WINDOW,
    "early_ratio": EARLY_RATIO,
    "mid_ratio": MID_RATIO,
    "early_high": EARLY_HIGH_THRES,
    "early_low": EARLY_LOW_THRES,
    "mid_high": MID_HIGH_THRES,
    "mid_low": MID_LOW_THRES,
    "late_low": LATE_LOW_THRES,
})

{'k_init': 2, 'k_min': 1, 'k_max': 4, 'window': 4, 'early_ratio': 0.3, 'mid_ratio': 0.7, 'early_high': 0.75, 'early_low': 0.45, 'mid_high': 0.85, 'mid_low': 0.5, 'late_low': 0.55}


In [ ]:
# 实现分阶段 adaptive controller，结合生成进度和 acceptance 趋势动态调整 k
def update_phase_aware_k(
    k_current,
    acceptance_history,
    generated_len,
    max_new_tokens,
    k_min=PHASE_K_MIN,
    k_max=PHASE_K_MAX,
    window=PHASE_WINDOW,
    early_ratio=EARLY_RATIO,
    mid_ratio=MID_RATIO,
    early_high=EARLY_HIGH_THRES,
    early_low=EARLY_LOW_THRES,
    mid_high=MID_HIGH_THRES,
    mid_low=MID_LOW_THRES,
    late_low=LATE_LOW_THRES,
):
    if len(acceptance_history) == 0:
        return k_current

    progress = generated_len / max_new_tokens if max_new_tokens > 0 else 1.0
    recent = acceptance_history[-window:]
    avg_recent = sum(recent) / len(recent)

    # 最近两轮 acceptance，用于判断是否允许升 k
    last_two = acceptance_history[-2:] if len(acceptance_history) >= 2 else acceptance_history

    # 早期阶段：允许适度探索
    if progress < early_ratio:
        if avg_recent < early_low:
            return max(k_current - 1, k_min)
        if len(last_two) >= 2 and min(last_two) >= 0.8 and avg_recent >= early_high:
            return min(k_current + 1, k_max)
        return k_current

    # 中期阶段：更谨慎
    elif progress < mid_ratio:
        if avg_recent < mid_low:
            return max(k_current - 1, k_min)
        if len(last_two) >= 2 and min(last_two) >= 0.85 and avg_recent >= mid_high:
            return min(k_current + 1, k_max)
        return k_current

    # 后期阶段：原则上不再主动升 k，只在不稳定时收缩
    else:
        if avg_recent < late_low:
            return max(k_current - 1, k_min)
        return k_current

In [ ]:
# 实现分阶段自适应 draft 控制器的单条生成函数，记录 k 和 acceptance 的动态变化
@torch.no_grad()
def generate_phase_aware_adaptive_k(
    article: str,
    k_init: int = PHASE_K_INIT,
    k_min: int = PHASE_K_MIN,
    k_max: int = PHASE_K_MAX,
    window: int = PHASE_WINDOW,
    max_new_tokens: int = MAX_NEW_TOKENS,
):
    prompt = build_summary_prompt(article)

    inputs = target_tokenizer(prompt, return_tensors="pt").to(target_model.device)
    input_ids = inputs["input_ids"]

    generated = []
    total_accepted = 0
    total_drafted = 0

    k_current = k_init
    acceptance_history = []
    k_history = []

    start = time.perf_counter()

    while len(generated) < max_new_tokens:
        k_history.append(k_current)

        # 1) draft model 先生成 k_current 个 token
        draft_inputs = input_ids.to(draft_model.device)
        draft_tokens = draft_generate_k_tokens(draft_model, draft_inputs, k=k_current)
        total_drafted += k_current

        # 2) target model 验证
        accepted, accepted_count, next_target_token = verify_draft_tokens(
            target_model,
            input_ids.to(target_model.device),
            draft_tokens.to(target_model.device)
        )

        # 3) 写入被接受的 token
        for tok in accepted:
            tok_tensor = torch.tensor([[tok]], device=target_model.device)
            input_ids = torch.cat([input_ids.to(target_model.device), tok_tensor], dim=1)
            generated.append(tok)

            if len(generated) >= max_new_tokens:
                break

        total_accepted += accepted_count

        round_acceptance = accepted_count / k_current if k_current > 0 else 0.0
        acceptance_history.append(round_acceptance)

        if len(generated) >= max_new_tokens:
            break

        # 4) 写入 target 的下一个真实 token
        input_ids = torch.cat([input_ids.to(target_model.device), next_target_token], dim=1)
        generated.append(int(next_target_token.item()))

        if int(next_target_token.item()) == target_tokenizer.eos_token_id:
            break

        # 5) 分阶段更新 k
        k_current = update_phase_aware_k(
            k_current=k_current,
            acceptance_history=acceptance_history,
            generated_len=len(generated),
            max_new_tokens=max_new_tokens,
            k_min=k_min,
            k_max=k_max,
            window=window,
        )

    end = time.perf_counter()

    output_ids = generated[:max_new_tokens]
    output_text = target_tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    stop_markers = ["Human:", "Assistant:", "\n\nHuman:", "\n\nAssistant:"]
    for marker in stop_markers:
        if marker in output_text:
            output_text = output_text.split(marker)[0].strip()

    latency = end - start
    output_tokens = len(output_ids)
    acceptance_rate = total_accepted / total_drafted if total_drafted > 0 else 0.0
    avg_k = sum(k_history) / len(k_history) if len(k_history) > 0 else 0.0

    return {
        "prompt": prompt,
        "prediction": output_text,
        "latency_sec": latency,
        "output_tokens": output_tokens,
        "tokens_per_sec": output_tokens / latency if latency > 0 else None,
        "acceptance_rate": acceptance_rate,
        "accepted_tokens": total_accepted,
        "drafted_tokens": total_drafted,
        "avg_k": avg_k,
        "k_history": k_history,
        "acceptance_history": acceptance_history,
    }

In [ ]:
# 对单条样本测试分阶段 adaptive controller，观察 k 历史和 acceptance 变化
sample_article = df_data.iloc[0]["article"]
sample_ref = df_data.iloc[0]["reference"]

result_phase = generate_phase_aware_adaptive_k(sample_article)

print("=== Prediction (phase-aware adaptive-k) ===")
print(result_phase["prediction"])
print("\n=== Reference ===")
print(sample_ref)
print("\n=== Stats ===")
print({
    "latency_sec": round(result_phase["latency_sec"], 3),
    "output_tokens": result_phase["output_tokens"],
    "tokens_per_sec": round(result_phase["tokens_per_sec"], 3) if result_phase["tokens_per_sec"] else None,
    "acceptance_rate": round(result_phase["acceptance_rate"], 3),
    "accepted_tokens": result_phase["accepted_tokens"],
    "drafted_tokens": result_phase["drafted_tokens"],
    "avg_k": round(result_phase["avg_k"], 3),
})

print("\n=== k history ===")
print(result_phase["k_history"])

print("\n=== acceptance history ===")
print([round(x, 3) for x in result_phase["acceptance_history"]])

=== Prediction (phase-aware adaptive-k) ===
Zully Broussard's kidney donation led to a six-patient kidney transplant chain, facilitated by data processing of genetic profiles, surprising her with the potential to extend lives beyond her initial recipient. The process, made possible by David Jacobs' MatchGrid program, has since been streamlined, allowing for rapid matching of compatible

=== Reference ===
Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

=== Stats ===
{'latency_sec': 2.789, 'output_tokens': 64, 'tokens_per_sec': 22.944, 'acceptance_rate': 0.407, 'accepted_tokens': 24, 'drafted_tokens': 59, 'avg_k': 1.475}

=== k history ===
[2, 2, 3, 3, 3, 3, 2, 1, 1, 1, 1, 2, 3, 3, 3, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

=== acceptance history ===
[1.0, 1.0, 0.0, 0.667, 0.333, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.333, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.

In [ ]:
# 批量运行分阶段 adaptive controller，并保存逐样本结果
phase_aware_results = []

for idx, row in df_data.iterrows():
    out = generate_phase_aware_adaptive_k(row["article"])
    phase_aware_results.append({
        "id": row["id"],
        "reference": row["reference"],
        "prediction": out["prediction"],
        "latency_sec": out["latency_sec"],
        "output_tokens": out["output_tokens"],
        "tokens_per_sec": out["tokens_per_sec"],
        "acceptance_rate": out["acceptance_rate"],
        "accepted_tokens": out["accepted_tokens"],
        "drafted_tokens": out["drafted_tokens"],
        "avg_k": out["avg_k"],
    })

df_phase_aware = pd.DataFrame(phase_aware_results)
df_phase_aware.head(3)

In [ ]:
# 计算分阶段 adaptive controller 的整体质量与效率指标
rouge_scores_phase = rouge.compute(
    predictions=df_phase_aware["prediction"].tolist(),
    references=df_phase_aware["reference"].tolist(),
    use_stemmer=True
)

phase_aware_stats = {
    "avg_latency_sec": float(df_phase_aware["latency_sec"].mean()),
    "avg_output_tokens": float(df_phase_aware["output_tokens"].mean()),
    "avg_tokens_per_sec": float(df_phase_aware["tokens_per_sec"].mean()),
    "avg_acceptance_rate": float(df_phase_aware["acceptance_rate"].mean()),
    "avg_k": float(df_phase_aware["avg_k"].mean()),
    "rouge1": rouge_scores_phase["rouge1"],
    "rouge2": rouge_scores_phase["rouge2"],
    "rougeL": rouge_scores_phase["rougeL"],
    "rougeLsum": rouge_scores_phase["rougeLsum"],
}

phase_aware_stats

In [ ]:
# 整理 target-only、fixed-k、simple adaptive、phase-aware adaptive 的总对比表
comparison_rows = []

comparison_rows.append({
    "method": "target_only",
    "avg_latency_sec": summary_stats["avg_latency_sec"],
    "avg_tokens_per_sec": summary_stats["avg_tokens_per_sec"],
    "avg_acceptance_rate": None,
    "avg_k": None,
    "rouge1": summary_stats["rouge1"],
    "rouge2": summary_stats["rouge2"],
    "rougeL": summary_stats["rougeL"],
    "rougeLsum": summary_stats["rougeLsum"],
})

comparison_rows.append({
    "method": "fixed_k_2",
    "avg_latency_sec": fixed_k2_stats["avg_latency_sec"],
    "avg_tokens_per_sec": fixed_k2_stats["avg_tokens_per_sec"],
    "avg_acceptance_rate": fixed_k2_stats["avg_acceptance_rate"],
    "avg_k": 2.0,
    "rouge1": fixed_k2_stats["rouge1"],
    "rouge2": fixed_k2_stats["rouge2"],
    "rougeL": fixed_k2_stats["rougeL"],
    "rougeLsum": fixed_k2_stats["rougeLsum"],
})

comparison_rows.append({
    "method": "adaptive_k_simple",
    "avg_latency_sec": adaptive_stats["avg_latency_sec"],
    "avg_tokens_per_sec": adaptive_stats["avg_tokens_per_sec"],
    "avg_acceptance_rate": adaptive_stats["avg_acceptance_rate"],
    "avg_k": adaptive_stats["avg_k"],
    "rouge1": adaptive_stats["rouge1"],
    "rouge2": adaptive_stats["rouge2"],
    "rougeL": adaptive_stats["rougeL"],
    "rougeLsum": adaptive_stats["rougeLsum"],
})

comparison_rows.append({
    "method": "adaptive_k_phase_aware",
    "avg_latency_sec": phase_aware_stats["avg_latency_sec"],
    "avg_tokens_per_sec": phase_aware_stats["avg_tokens_per_sec"],
    "avg_acceptance_rate": phase_aware_stats["avg_acceptance_rate"],
    "avg_k": phase_aware_stats["avg_k"],
    "rouge1": phase_aware_stats["rouge1"],
    "rouge2": phase_aware_stats["rouge2"],
    "rougeL": phase_aware_stats["rougeL"],
    "rougeLsum": phase_aware_stats["rougeLsum"],
})

df_phase_compare = pd.DataFrame(comparison_rows)
df_phase_compare